---
title: "DRG Cleaning"

author: "Carlos Resurreccion"

date: "2024-07-01"

---

## Parameters

In [25]:
# IMPORTANT PARAMETERS:
year_to_load <- "2018" # Which claims year to load # TODO: maybe add a script that loops through all claims?
split_parts <- 5 # How many parts to split the 12+m row claims file into
rows_to_show <- 10 # How many rows/entries to show in summary tables

# Input:
to_read <- FALSE # TODO: Deprecated and Unused # Whether to forcibly read the whole file again instead of using the split parts created even if available
to_split <- TRUE # TODO: Deprecated, only used when to_sample is TRUE # Whether to split into split_parts parts (i.e. to fit in 32gb RAM). 
to_sample <- TRUE # Whether to sample each split_parts part by sample_size_divisor (useful when iterating through code runs in quick succession)
sample_size_divisor <- 125 # Sample size divisor: Formula for sample size is total_rows / split_parts / sample_size_divisor

# Output:
to_write <- TRUE # Whether to write out intermediate files and caches (i.e. part files, sample files). TODO: upload to BQ as well
to_group <- TRUE # Whether to export for the batch grouper or not

# Debug:
to_profvis <- TRUE # Conduct runtime duration analysis via profvis or not
to_view_checks <- TRUE # Whether to view checks and print statements
to_view_checks_parallelized <- FALSE # Whether to view intermediate per part/chunk checks and print statements (not consolidated) when parallelized
to_parallelize <- TRUE # Whether to parallelize each split_parts part into availableCores() - 1 chunks. Cuts down processing time from 120min to 15min.
intermediate_rows_to_show <- Inf # Per part/chunk rows_to_show (leave at Inf)

drop_cols <- c( # Which columns to drop
  paste0("ICDCODE", 13:14), # Start
  "ICCODED15", # note that ICDCODE15 is misspelled as ICCODED15 in all claims
  paste0("ICDCODE", 16:170) # Continuation
)

seed <- 123 # Seed for reproducibility (Important for stuff like randomly choosing a pdx among multiple possible options)
set.seed(seed) # Setting the seed
global_seed <- seed # global_seed for future_lapply parts for parallelized operations

options(future.globals.maxSize = 1024 * 1024^2) # Allowing each future_lapply session to use more memory


## Load Required Libraries & Initial Functions

In [26]:
options(verbose = FALSE) # Hide verbose output for script and library loading
options(warn = -1) # Hide warnings for script sourcing and library loading
library(here) # Library here() so scipts can be loaded


In [27]:
scripts <- list( # List of scripts to source
  libraries = "00_libraries.R",
  formats = "01_data-formats.R",
  paths = "02_file-paths.R",
  general = "03_general-functions.R",
  clean = "04a_clean-data-functions.R",
  chunk = "04b_chunk-functions.R",
  part = "04c_part-functions.R",
  io = "05_io-functions.R",
  icd = "06_icd-functions.R",
  rvs = "07_rvs-functions.R",
  pdx = "08_pdx-functions.R",
  grouper = "09_grouper-functions.R",
  timing = "10_timing-functions.R",
  debug = "11_debug-functions.R",
  summary = "12_summary-functions.R"
)

# Loop to source above scripts
for (script in scripts) source(here("data-cleaning/r_scripts", script))


Total Rows via cached object: 11777674Total Rows via cached object: 11777674

In [28]:
# TODO: figure out a way to return to default outputs
# since verbose = TRUE is way too verbose compared to default
options(warn = 1) # Reenable warnings; see above comments


## Load Mapping Data

In [29]:
# Read in all rvs codes and turn to character for further processing
proc <- fread(here(excel_path, "proc.csv"))
proc[, CODE := as.character(CODE)]

# Read in icd9cm equivalents of rvs codes
rvs_icd9 <- fread(here(aux_path, "rvs_icd9cm.csv"),
  select = c("rvs", "icd9cm")
)

# Convert to character for further processing
rvs_icd9[, rvs := as.character(rvs)]

# Convert to character and also remove decimals
# whilst keeping trailing zeroes
rvs_icd9[, icd9cm := as.character(icd9cm * 100)]

# Merge with proc from above, to be able to classify by DRGUSE
rvs_icd9 <- merge(rvs_icd9, proc[, .(CODE, DRGUSE)],
  by.x = "icd9cm", by.y = "CODE", all.x = TRUE
)

# Filter by DRGUSE
rvs_icd9[, is_drg := !is.na(DRGUSE) & DRGUSE]

# Remove DRGUSE and filter out NAs
rvs_icd9 <- rvs_icd9[!is.na(rvs) & !is.na(icd9cm), -"DRGUSE"]

# Read in PHIC all case rates
acr_rvs <- fread(here(aux_path, "acr_rvs.csv"))

# Read in the thai icd10 library
tdrg_icd10 <- fread(here(aux_path, "i10.csv"))

# Set the key if not already set
setkey(tdrg_icd10, "CODE")

# Subset and assign the result to acc_pdx
acc_pdx <- unique(tdrg_icd10[ACCPDX == "Y", CODE])


## Read, Process, Export Data (Looping through all parts)

In [30]:
all_parts_summaries <- list() # initialize list for summaries
processing_times <- numeric(split_parts) # initialize list for ETA

num_cores <- availableCores() - 1 # detect number of cores available for parallelization

# Define a codeblock to avoid repeating it twice when to_profvis is TRUE and again if FALSE
# Makes it easier to maintain as well, since we only need to modify one section instead of two
unified_block <- function() {
  # Start main execution logic
  split_and_save_parts() # Read, split, and save partial files

  if (to_parallelize) {
    # Start the parallelization session
    plan(multisession, workers = num_cores)
  } else {
    # Status quo; i.e. remain sequential
    plan(sequential)
  }

  # For each partial file in N (split_parts) files,
  for (part in 1:split_parts) {
    # Process the partial file with or without parallelization
    result <- process_part(
      part, num_cores, to_view_checks, global_seed,
      intermediate_rows_to_show, rvs_icd9, tdrg_icd10,
      acc_pdx, to_parallelize, to_write, to_group, to_sample
    )

    # Save partial summaries to a list
    all_parts_summaries[[part]] <- result$combined_summary

    # Save partial processing time to a list
    processing_times[part] <- result$processing_time

    # Print status update and ETA
    # - VS Code: Updates are shown after complete execution
    # - Positron: Updates are shown live
    # - JupyterLab: Untested
    print_status_update(part, split_parts, processing_times)

    # Save resulting partial dt to variable dt, to be used for
    # Speed calculations
    dt <- result$dt
  }
  
  # Summaries are consolidated from 5 split_parts * 15 chunks = 75 sub outputs
  print_summary_tables( # Print final summaries
    combine_parts_summaries( # input
      all_parts_summaries, intermediate_rows_to_show # input cont'd.
    ), rows_to_show # param
  )

  # End main execution logic and return(dt)
  plan(sequential) # end parallelization
  return(dt) 
}

if (to_profvis) saveWidget(profvis({dt <- unified_block()}), here(profvis_path)) else dt <- unified_block()


Status Update
Finished: Part 1 of 5
Status Update
Finished: Part 1 of 5


Elapsed: 6 seconds
ETA: 23 seconds
Elapsed: 6 seconds
ETA: 23 seconds


Status Update
Finished: Part 2 of 5
Elapsed: 10 seconds
ETA: 15 seconds
Status Update
Finished: Part 2 of 5
Elapsed: 10 seconds
ETA: 15 seconds


Status Update
Finished: Part 3 of 5
Elapsed: 16 seconds
ETA: 11 seconds
Status Update
Finished: Part 3 of 5
Elapsed: 16 seconds
ETA: 11 seconds


Status Update
Finished: Part 4 of 5
Elapsed: 20 seconds
ETA: 5 seconds
Status Update
Finished: Part 4 of 5
Elapsed: 20 seconds
ETA: 5 seconds


Status Update
Finished: Part 5 of 5
Elapsed: 27 seconds
ETA: 0 seconds


Rename Success:
Status Update
Finished: Part 5 of 5
Elapsed: 27 seconds
ETA: 0 seconds


Rename Success:


 TRUE 



Table: ICD Replacements 1

|old_code |new_code | count|
|:--------|:--------|-----:|
|J18.92   |J1892    |  5917|
|A09.9    |A099     |  3290|
|N39.0    |N390     |  2686|
|A97.1    |A971     |  1468|
|K29.1    |K291     |  1054|
|J45.90   |J4590    |  1016|
|I10.1    |I101     |   977|
|A97.0    |A970     |   911|
|I10.9    |I109     |   877|
|P36.9    |P369     |   765|


Table: ICD Replacements 2

|old_code |new_code | count|
|:--------|:--------|-----:|
|I21.9    |I219     |    46|
|E86.1    |E861     |    24|
|I21.4    |I214     |    19|
|I63.9    |I639     |    15|
|O75.8    |O758     |    13|
|I61.9    |I619     |    10|
|I21.0    |I210     |     4|
|O75.1    |O751     |     4|
|N39.0    |N390     |     3|
|I60.9    |I609     |     3|


Patient Type Unmapped: NULL

Memcat Parent Unmapped: NULL

Memcat Child Unmapped: NULL

Discharge Unmapped: NULL



Table: Discarded RVS Codes One

|CODE  | count|
|:-----|-----:|
|77401 |   971|
|77418 |   487|
|90375 |   418|
|77421 |

## Runtime Estimation

### Stop Timer & Calculate Speed

In [31]:
print_time_estimates() # Print time estimates along with estimate for full claims file


Time spent (total)               : 29.27 sec elapsed
Time spent (t/row) for 94.2k rows: 0.31 msec
Time (est) (total) for 11.8m rows: 60.98 min
Time spent (total)               : 29.27 sec elapsed
Time spent (t/row) for 94.2k rows: 0.31 msec
Time (est) (total) for 11.8m rows: 60.98 min


## Debugging

In [32]:
# in case we want to run this cell independently:
library(here)
source(here("data-cleaning/r_scripts", scripts$debug))

# Consolidate all r_scripts scripts into everything.R; useful for debugging
concatenate_r_files(here("data-cleaning/r_scripts"), here("data-cleaning/everything/everything.R"))


To extract all code portions of this ipynb file (run in VS Code terminal):

jupyter nbconvert --no-prompt --to script data-cleaning/drg-cleaning.ipynb --output everything/drg-cleaning

